# SQL Agent with Walmart Sales
**Goal:** Create a basic SQL agent to interact with the Walmart Sales database

## Libraries

In [ ]:
# Most Important: AI
from langchain_openai import ChatOpenAI
from langchain_classic.chains import create_sql_query_chain
from langchain_community.utilities import SQLDatabase

# Data Science
import pandas as pd
import sqlalchemy as sql

# Utilities
import os
import yaml
import re
from pprint import pprint
from IPython.display import Markdown

## AI Setup

In [ ]:
os.environ["OPENAI_API_KEY"] = yaml.safe_load(open('../credentials.yml'))['openai']

OPENAI_LLM = "gpt-4o-mini"

## 1.0 Database Setup — Walmart Sales

In [ ]:
PATH_DB = "sqlite:///../data/walmart_sales.db"

sql_engine = sql.create_engine(PATH_DB)
conn = sql_engine.connect()

# Show all tables
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

## 2.0 Connect LangChain to the Database

In [ ]:
db = SQLDatabase.from_uri(PATH_DB)

print("Dialect:", db.dialect)
print("Tables:", db.get_usable_table_names())
print("\nSample data:")
print(db.run("SELECT * FROM daily_demand LIMIT 5;"))

## 3.0 Create the SQL Query Chain (Agent)

In [ ]:
model = ChatOpenAI(
    model=OPENAI_LLM,
    temperature=0.7,
)

# Create the SQL query chain
chain = create_sql_query_chain(model, db)
chain

In [ ]:
response = chain.invoke({'question': "What are the top 5 items by total demand value?"})
pprint(response)
Markdown(response)

## 4.0 SQL Parsing Utility

In [ ]:
def extract_sql_code(text: str):
    """
    Extracts the SQL query from a block of text.
    """
    patterns = [
        r"SQLQuery:\s*```sql\s*(?P<sql>[\s\S]+?)```",
        r"```sql\s*(?P<sql>[\s\S]+?)```",
        r"```(?:[\s\S]*?)\s*(?P<sql>SELECT[\s\S]+?)```",
        r"SQLQuery:\s*(?P<sql>[\s\S]+?)(?=\n\s*\n|$)",
        r"(?P<sql>SELECT[\s\S]+?;)(?=\s|$)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            sql_code = m.group("sql").strip()
            if sql_code.startswith(('"', "'")) and sql_code.endswith(('"', "'")):
                sql_code = sql_code[1:-1].strip()
            return sql_code
    return None

pprint(extract_sql_code(response))
Markdown(f"```sql\n{extract_sql_code(response)}\n```")

In [ ]:
# Run extracted SQL against DB
pprint(db.run(extract_sql_code(response)))

## 5.0 Additional Questions

In [ ]:
# What are total monthly sales?
q = chain.invoke({'question': "What are the total sales value by year-month? Order by date."})
sql_q = extract_sql_code(q)
pprint(sql_q)
pd.read_sql(sql_q, conn)

In [ ]:
# Top items by date range
q2 = chain.invoke({'question': "Which 10 items have the highest average daily demand value?"})
sql_q2 = extract_sql_code(q2)
pprint(sql_q2)
pd.read_sql(sql_q2, conn)